# 3D U-Net on BraTS2020 — **Colab edition**

Companion to `brats_3d_unet.ipynb`, which is the **Kaggle** edition. Same model,
same loss, same splits — only the environment differs. Nothing here auto-detects
anything: paths are fixed for Colab, so there is no chance of writing
checkpoints somewhere that disappears.

**Why the split.** An earlier single notebook tried to detect its platform. On
Colab that detection was wrong — the Colab image provides a `/kaggle` path, so
it believed it was on Kaggle and pointed checkpoints at `/kaggle/working`, which
does **not** persist on a Colab VM. Everything would have been lost on
disconnect. Two explicit notebooks remove the guess entirely.

**Baseline to beat:** mean tumour Dice **0.76**, enhancing tumour **0.84**.

---

### What persists here

| Path | Survives disconnect? |
|---|---|
| `/content/...` (VM disk) — including the dataset | **No** |
| `/content/drive/MyDrive/mri_3d_unet/` | **Yes — it is your Drive** |

Checkpoints are written to Drive every epoch. Losing a session costs the current
epoch, never the run.

### Colab's limit, stated plainly

Free Colab stops on idle after roughly 90 minutes without browser interaction.
**You will not get 60 epochs overnight here.** You will get a few hours of
progress, cleanly checkpointed, and can continue next session.

For long unattended runs use the Kaggle notebook — *Save & Run All (Commit)*
keeps going up to 12 hours after you close the tab. Checkpoints move between the
two through the Kaggle Dataset in the last cell.

## 1 · GPU

*Runtime → Change runtime type → T4 GPU* if this reports no GPU.

Anything with **12 GB or more** runs the settings below unchanged. On a smaller
card, set `PATCH = 96` in cell 4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU  : {p.name}  {p.total_memory/1e9:.1f} GB")
    if p.total_memory / 1e9 < 12:
        print("  ! under 12 GB — set PATCH = 96 in cell 4")
else:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU.")

## 2 · Mount Drive

This is the cell that makes the run survive a disconnect. Checkpoints go to
Drive, never to the VM's disk.

`CKPT_DIR` is hardcoded — no detection, no ambiguity.

In [ ]:
import os, glob, json, random, time
from google.colab import drive

drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/mri_3d_unet'
os.makedirs(CKPT_DIR, exist_ok=True)

# Fail loudly rather than silently training into a directory that will vanish.
assert CKPT_DIR.startswith('/content/drive/'), "checkpoints must live on Drive"
assert os.path.ismount('/content/drive') or os.path.exists('/content/drive/MyDrive'), \
    "Drive did not mount — re-run this cell before training"

print("checkpoints ->", CKPT_DIR)
print("existing    :", sorted(os.listdir(CKPT_DIR)) or "(none — first run)")

## 3 · Get BraTS2020

Pulled straight from Kaggle into the VM, which runs at datacentre speed —
**do not upload 14 GB from your own machine.**

### Authenticating — use Colab Secrets, not a file

1. kaggle.com → Settings → **API Tokens** → *Generate New Token*. Copy the
   `KGAT_...` string.
2. In Colab, click the **key icon** in the left sidebar (Secrets).
3. *Add new secret* → name it exactly `KAGGLE_API_TOKEN`, paste the token as the
   value, and enable **Notebook access**.

The token then never appears in a cell, in the notebook file, or in your
screenshots — which matters, because a leaked token lets anyone act as your
account. If you have ever pasted one into a cell or shared an image of one,
revoke it and generate a new one.

Older `kaggle.json` credentials still work and are handled as a fallback below.

This cell re-runs each session because the VM disk is wiped. It takes a few
minutes and costs nothing but time — your checkpoints on Drive are unaffected.

In [ ]:
ROOT = '/content/data/MICCAI_BraTS2020_TrainingData'

if not os.path.exists(ROOT):
    authed = False

    # Preferred: the new KGAT_ token, held in Colab Secrets (key icon, sidebar).
    # Never put the token in a cell — it ends up in the saved notebook.
    try:
        from google.colab import userdata
        tok = userdata.get('KAGGLE_API_TOKEN')
        if tok:
            os.environ['KAGGLE_API_TOKEN'] = tok
            authed = True
            print("authenticated via Colab secret KAGGLE_API_TOKEN")
    except Exception as e:
        print("no Colab secret found:", type(e).__name__)

    # Fallback: legacy kaggle.json credentials
    if not authed:
        if os.path.exists('/root/.kaggle/kaggle.json'):
            authed = True
            print("authenticated via existing /root/.kaggle/kaggle.json")
        else:
            print("Upload legacy kaggle.json, or press Cancel and use a Colab "
                  "secret named KAGGLE_API_TOKEN instead (see the text above).")
            from google.colab import files
            up = files.upload()
            if up:
                os.makedirs('/root/.kaggle', exist_ok=True)
                os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
                os.chmod('/root/.kaggle/kaggle.json', 0o600)
                authed = True

    assert authed, ("No Kaggle credentials. Add a Colab secret named "
                    "KAGGLE_API_TOKEN (key icon in the sidebar) and re-run.")

    !pip install -q kaggle
    !kaggle datasets download -d awsaf49/brats20-dataset-training-validation -p /content --unzip -q
    !mkdir -p /content/data
    !mv /content/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData /content/data/ 2>/dev/null || true

assert os.path.exists(ROOT), (
    f"dataset not found at {ROOT} — check the download step above for errors")
cases = sorted(d for d in os.listdir(ROOT) if d.startswith('BraTS20'))
print(f"{len(cases)} cases at {ROOT}")

## 3b · (Optional) Continue a run started on Kaggle

Skip unless you are **handing over** — cell 7 already finds anything in Drive.

Set `CKPT_DATASET` to the Kaggle Dataset slug you pushed from the Kaggle
notebook's last cell, and this pulls those checkpoints into Drive so training
continues at the same epoch.

In [ ]:
CKPT_DATASET = ''      # e.g. 'yourname/mri-3d-checkpoints' — '' to skip

if CKPT_DATASET:
    !pip install -q kaggle
    !kaggle datasets download -d {CKPT_DATASET} -p {CKPT_DIR} --unzip -q
    print("pulled into Drive:", sorted(os.listdir(CKPT_DIR)))
else:
    print("skipped — using whatever is already in Drive")

## 4 · Configuration

`PATCH = 128` means training on random 128³ crops rather than whole
240×240×155 volumes. Full volumes will not fit, and patches are standard for 3D
medical segmentation — they also act as augmentation, since each epoch sees
different crops.

**Splitting is at patient level, never at patch level.** Patches from one patient
are highly correlated, so splitting below the patient leaks information between
train and validation and inflates every score. Same rule as the 2D work, which
is what makes the comparison fair.

Keep these identical to the Kaggle notebook, or the two are no longer the same
experiment.

In [ ]:
PATCH       = 128     # 96 if your GPU has under 12 GB
BATCH       = 1       # 3D patches are large; accumulate rather than batch
ACCUM       = 2       # effective batch = BATCH * ACCUM
BASE_FILT   = 16
LR          = 1e-3
EPOCHS      = 60      # total across ALL sessions, not per session
N_CASES     = 126     # match the 2D run so the comparison is like for like
VAL_FRAC    = 0.2
SEED        = 42

# BraTS labels: 0 bg, 1 necrotic, 2 oedema, 4 enhancing. Label 3 does not exist
# in the raw data, so 4 is remapped to 3 — CrossEntropyLoss needs contiguous
# class indices.
NUM_CLASSES = 4
MODALITIES  = ['t1', 't1ce', 't2', 'flair']

import numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"patch {PATCH}^3 | effective batch {BATCH*ACCUM} | {EPOCHS} epochs total")

## 5 · Dataset

Each item is a random patch from one patient, all four modalities stacked as
channels — the same 4-channel input the 2D model used, for the same reason: each
sequence reveals a different part of the tumour (T1c the enhancing rim, FLAIR/T2
the oedema).

Patches are **biased towards tumour-containing regions**. Without that, most
random 128³ crops from a 240×240×155 volume contain no tumour at all and the
model spends the run learning to predict background.

In [ ]:
import nibabel as nib
from torch.utils.data import Dataset, DataLoader

def norm(v):
    b = v[v > 0]
    if b.size == 0:
        return v.astype(np.float32)
    return np.clip((v - b.mean()) / (b.std() + 1e-8), -5, 5).astype(np.float32)

class BraTS3D(Dataset):
    def __init__(self, case_dirs, patch=PATCH, train=True, samples=2):
        self.dirs, self.patch, self.train = case_dirs, patch, train
        self.samples = samples if train else 1

    def __len__(self):
        return len(self.dirs) * self.samples

    def __getitem__(self, i):
        d = self.dirs[i // self.samples]
        cid = os.path.basename(d)
        vols = [norm(nib.load(f"{d}/{cid}_{m}.nii").get_fdata()) for m in MODALITIES]
        seg = nib.load(f"{d}/{cid}_seg.nii").get_fdata().astype(np.int64)
        seg[seg == 4] = 3
        x = np.stack(vols)
        p = self.patch

        if self.train:
            fg = np.argwhere(seg > 0)
            if len(fg) and np.random.rand() < 0.7:
                c = fg[np.random.randint(len(fg))]
                st = [int(np.clip(c[k] - p // 2, 0, max(seg.shape[k] - p, 0)))
                      for k in range(3)]
            else:
                st = [np.random.randint(0, max(seg.shape[k] - p, 1)) for k in range(3)]
        else:
            st = [max((seg.shape[k] - p) // 2, 0) for k in range(3)]

        sl = tuple(slice(st[k], st[k] + p) for k in range(3))
        x, y = x[(slice(None),) + sl], seg[sl]
        pad = [(0, max(p - x.shape[k + 1], 0)) for k in range(3)]
        if any(b for _a, b in pad):
            x = np.pad(x, [(0, 0)] + pad); y = np.pad(y, pad)
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

dirs = [os.path.join(ROOT, c) for c in cases[:N_CASES]]
dirs = [d for d in dirs if os.path.exists(f"{d}/{os.path.basename(d)}_seg.nii")]
random.Random(SEED).shuffle(dirs)                     # PATIENT-level split
n_val = max(1, int(len(dirs) * VAL_FRAC))
val_dirs, train_dirs = dirs[:n_val], dirs[n_val:]

train_dl = DataLoader(BraTS3D(train_dirs, train=True), batch_size=BATCH,
                      shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(BraTS3D(val_dirs, train=False), batch_size=1, num_workers=2)
print(f"{len(train_dirs)} train patients / {len(val_dirs)} val patients "
      f"({len(train_dl)} train patches per epoch)")

## 6 · Model and loss

**Loss = Cross-Entropy + soft Dice**, identical to the 2D run.

Dice is not optional. Background is **99.03 %** of voxels in BraTS, so a model
trained on cross-entropy alone reaches 99 % accuracy by predicting "background"
everywhere and finding no tumour at all. Dice measures region overlap, so that
degenerate solution scores zero.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

def block(i, o):
    return nn.Sequential(
        nn.Conv3d(i, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True),
        nn.Conv3d(o, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True))

class UNet3D(nn.Module):
    """Same shape as our 2D model: encoder, bottleneck, decoder, skip
    connections. Skips carry fine boundary detail straight across, which keeps
    tumour edges sharp — in medicine the boundary is the finding."""
    def __init__(self, in_ch=4, n_cls=NUM_CLASSES, f=BASE_FILT):
        super().__init__()
        self.e1, self.e2, self.e3 = block(in_ch, f), block(f, f*2), block(f*2, f*4)
        self.bott = block(f*4, f*8)
        self.u3 = nn.ConvTranspose3d(f*8, f*4, 2, 2); self.d3 = block(f*8, f*4)
        self.u2 = nn.ConvTranspose3d(f*4, f*2, 2, 2); self.d2 = block(f*4, f*2)
        self.u1 = nn.ConvTranspose3d(f*2, f,   2, 2); self.d1 = block(f*2, f)
        self.out = nn.Conv3d(f, n_cls, 1)
        self.pool = nn.MaxPool3d(2)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        b  = self.bott(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b),  e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)                            # raw logits

def dice_loss(logits, target, eps=1.0):
    p = F.softmax(logits, 1)
    t = F.one_hot(target, NUM_CLASSES).permute(0, 4, 1, 2, 3).float()
    dims = (0, 2, 3, 4)
    inter = (p * t).sum(dims); denom = p.sum(dims) + t.sum(dims)
    return 1 - ((2 * inter + eps) / (denom + eps))[1:].mean()   # ignore background

ce = nn.CrossEntropyLoss()
def criterion(lg, t): return ce(lg, t) + dice_loss(lg, t)

dev = 'cuda'
model = UNet3D().to(dev)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters "
      f"(the 2D model has 7.77 M)")

## 7 · Resume

Finds the newest checkpoint in Drive and restores weights, optimiser state,
epoch counter and metric history — including checkpoints that came from the
Kaggle notebook.

First run: "starting fresh". Every run after: the epoch it continues from.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler('cuda')
start_epoch, history, best_dice = 0, [], 0.0

found = sorted(glob.glob(f"{CKPT_DIR}/epoch_*.pt")) or \
        ([f"{CKPT_DIR}/best.pt"] if os.path.exists(f"{CKPT_DIR}/best.pt") else [])
if found:
    ck = torch.load(found[-1], map_location=dev)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    if 'scaler' in ck: scaler.load_state_dict(ck['scaler'])
    start_epoch = ck['epoch'] + 1
    history     = ck.get('history', [])
    best_dice   = ck.get('best_dice', 0.0)
    print(f"RESUMING from {os.path.basename(found[-1])} -> epoch {start_epoch}")
    print(f"best mean tumour Dice so far: {best_dice:.4f}")
    print(f"previous session ran on: {ck.get('platform', 'unknown')}")
else:
    print("starting fresh (no checkpoint in Drive)")
print(f"will train epochs {start_epoch} -> {EPOCHS-1}")

## 8 · Train

Safe to interrupt at any point — the previous epoch is already in Drive.

**Keep the tab active.** Free Colab stops on idle after roughly 90 minutes, so
expect a few hours per session rather than the whole run. That is the reason the
Kaggle notebook exists.

Per-class Dice is reported every epoch, so you can watch the enhancing-tumour
class specifically — clinically the most important, and the one our 2D model
scores best on (0.84).

In [ ]:
CLASSES = ['necrotic', 'oedema', 'enhancing']

@torch.no_grad()
def validate():
    model.eval()
    inter = np.zeros(NUM_CLASSES); denom = np.zeros(NUM_CLASSES); losses = []
    for x, y in val_dl:
        x, y = x.to(dev), y.to(dev)
        with torch.amp.autocast('cuda'):
            lg = model(x); losses.append(criterion(lg, y).item())
        pr = lg.argmax(1)
        for c in range(NUM_CLASSES):
            pc, tc = (pr == c), (y == c)
            inter[c] += (pc & tc).sum().item()
            denom[c] += pc.sum().item() + tc.sum().item()
    return float(np.mean(losses)), np.where(denom > 0, 2*inter/np.maximum(denom, 1), np.nan)

for ep in range(start_epoch, EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0
    opt.zero_grad(set_to_none=True)
    for i, (x, y) in enumerate(train_dl):
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.amp.autocast('cuda'):
            loss = criterion(model(x), y) / ACCUM
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM == 0:
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        tot += loss.item() * ACCUM

    tr = tot / max(len(train_dl), 1)
    vl, dice = validate()
    mean_tumour = float(np.nanmean(dice[1:]))
    history.append({'epoch': ep, 'train_loss': tr, 'val_loss': vl,
                    'dice': [None if np.isnan(d) else float(d) for d in dice],
                    'mean_tumour_dice': mean_tumour, 'platform': 'colab'})

    per = "  ".join(f"{n} {dice[i+1]:.3f}" for i, n in enumerate(CLASSES))
    print(f"epoch {ep:3d} | train {tr:.4f} | val {vl:.4f} | "
          f"mean tumour Dice {mean_tumour:.4f} | {per} | {time.time()-t0:.0f}s", flush=True)

    ck = {'model': model.state_dict(), 'opt': opt.state_dict(),
          'scaler': scaler.state_dict(), 'epoch': ep, 'history': history,
          'best_dice': max(best_dice, mean_tumour), 'platform': 'colab',
          'config': {'patch': PATCH, 'base_filters': BASE_FILT,
                     'n_cases': N_CASES, 'seed': SEED}}
    torch.save(ck, f"{CKPT_DIR}/epoch_{ep:03d}.pt")
    if mean_tumour > best_dice:
        best_dice = mean_tumour
        torch.save(ck, f"{CKPT_DIR}/best.pt")
        print(f"   new best ({best_dice:.4f}) -> best.pt", flush=True)

    for f in sorted(glob.glob(f"{CKPT_DIR}/epoch_*.pt"))[:-3]:   # keep 3 newest
        os.remove(f)
    with open(f"{CKPT_DIR}/history.json", 'w') as f:
        json.dump(history, f, indent=2)

print(f"\ndone through epoch {EPOCHS-1}. best mean tumour Dice: {best_dice:.4f}")

## 9 · Did 3D beat 2D?

The comparison this notebook exists to make. Same dataset, same patient-level
split rule, same loss, same metric.

**One caveat, stated rather than buried.** Our 2D number is computed over whole
validation slices; this validates on centre patches. Close, but not an identical
evaluation — so treat a fraction of a point as noise. A real win should be
several points, and **if 3D does not clearly win, that is a legitimate result
worth reporting.**

In [ ]:
import matplotlib.pyplot as plt

hist = json.load(open(f"{CKPT_DIR}/history.json"))
ep = [h['epoch'] for h in hist]

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
a.plot(ep, [h['train_loss'] for h in hist], label='train')
a.plot(ep, [h['val_loss'] for h in hist], label='validation')
a.set_xlabel('epoch'); a.set_ylabel('loss'); a.legend(); a.grid(alpha=.3)
a.set_title('Learning curve — a growing gap means overfitting')

b.plot(ep, [h['mean_tumour_dice'] for h in hist], lw=2, label='3D U-Net (this run)')
b.axhline(0.76, ls='--', c='#b4432c', label='our 2D baseline (0.76)')
for i, n in enumerate(CLASSES):
    b.plot(ep, [h['dice'][i+1] for h in hist], alpha=.5, lw=1, label=n)
b.set_xlabel('epoch'); b.set_ylabel('Dice'); b.legend(fontsize=8); b.grid(alpha=.3)
b.set_title('Tumour Dice against the 2D result')
plt.tight_layout(); plt.savefig(f"{CKPT_DIR}/curves.png", dpi=130); plt.show()

best = max(h['mean_tumour_dice'] for h in hist)
d = best - 0.76
print(f"best 3D mean tumour Dice : {best:.4f}")
print(f"our 2D baseline          : 0.7600")
print(f"difference               : {d:+.4f}  "
      f"({'3D wins' if d > 0.02 else 'no clear win — report it as such'})")
print(f"epochs completed         : {len(hist)} / {EPOCHS}")
print("platforms used           :", sorted({h.get('platform','?') for h in hist}))

## 10 · Push so the Kaggle notebook can continue

Publishes `best.pt` and `history.json` as a Kaggle Dataset, which the Kaggle
notebook reads. Use it when handing a run the other way — Colab now, Kaggle
overnight.

Run at the **end of a session**, not every epoch: each push creates a new
dataset version.

In [ ]:
DATASET_SLUG = ''      # e.g. 'yourname/mri-3d-checkpoints'

if not DATASET_SLUG:
    print("set DATASET_SLUG to push. Skipping.")
else:
    !pip install -q kaggle
    stage = '/tmp/ckpt_push'; os.makedirs(stage, exist_ok=True)
    for f in ('best.pt', 'history.json', 'curves.png'):
        if os.path.exists(f"{CKPT_DIR}/{f}"):
            os.system(f'cp "{CKPT_DIR}/{f}" "{stage}/"')

    with open(f"{stage}/dataset-metadata.json", 'w') as f:
        json.dump({"title": DATASET_SLUG.split('/')[-1], "id": DATASET_SLUG,
                   "licenses": [{"name": "CC0-1.0"}]}, f)

    rc = os.system(f'kaggle datasets create -p {stage} -q 2>/dev/null')
    if rc != 0:
        last = json.load(open(f"{CKPT_DIR}/history.json"))[-1]
        os.system(f'kaggle datasets version -p {stage} '
                  f'-m "epoch {last["epoch"]}, dice {last["mean_tumour_dice"]:.4f}" -q')
    print(f"pushed to https://kaggle.com/datasets/{DATASET_SLUG}")
    print("In the Kaggle notebook, add this dataset and set CKPT_DATASET to the slug.")